# ARIMA Models: The Mathematical Foundation

*Part 5 of 8: Understanding AR, MA, and ARIMA*

Rachel's Holt-Winters worked... mostly. But traders needed **why**. They couldn't explain α, β, γ to clients.

Enter ARIMA: Mathematical rigor. Interpretability. Industry standard.

![ARIMA Components](arima_components_diagram.png)

**Note**: Part 5 = Theory. Part 6 = Implementation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 8)
np.random.seed(42)

## ARIMA = AR + I + MA

**AR (AutoRegressive)**: Value depends on past **values**  
**I (Integrated)**: Differencing to achieve stationarity  
**MA (Moving Average)**: Value depends on past **errors**

**ARIMA(p, d, q)** where:
- p = AR order
- d = differencing degree  
- q = MA order

## AR: Learning from Past Values

**AR(1)**: y(t) = φ₁·y(t-1) + ε(t)

- φ close to 1: Strong persistence
- φ close to 0: Weak persistence (white noise)
- Negative φ: Oscillating

In [ ]:
# Simulate AR(1) processes
n = 200

# Strong persistence
ar_high = ArmaProcess([1, -0.9], [1])
data_high = ar_high.generate_sample(nsample=n)

# Weak persistence  
ar_low = ArmaProcess([1, -0.3], [1])
data_low = ar_low.generate_sample(nsample=n)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(data_high, linewidth=1.5, color='#2E86AB')
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.3)
axes[0].set_title('AR(1), φ=0.9: Strong Persistence', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(data_low, linewidth=1.5, color='#06A77D')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.3)
axes[1].set_title('AR(1), φ=0.3: Weak Persistence', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## MA: Learning from Past Errors

**MA(1)**: y(t) = ε(t) + θ₁·ε(t-1)

Depends on forecast errors, not values.

![AR vs MA](ar_vs_ma_comparison.png)

In [ ]:
# Simulate MA(1)
ma_process = ArmaProcess([1], [1, 0.8])
ma_data = ma_process.generate_sample(nsample=n)

plt.figure(figsize=(16, 5))
plt.plot(ma_data, linewidth=1.5, color='#F18F01')
plt.axhline(y=0, color='red', linestyle='--', alpha=0.3)
plt.title('MA(1), θ=0.8: Spikier, Shorter Memory', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

print("Key difference:")
print("AR: Smooth, persistent (depends on past values)")
print("MA: Spiky, short memory (depends on past errors)")

## ACF & PACF: Your Tools

![ACF PACF Guide](acf_pacf_identification_table.png)

**Use patterns to identify model:**

| ACF | PACF | Model |
|-----|------|-------|
| Cuts off at q | Decays | MA(q) |
| Decays | Cuts off at p | AR(p) |
| Decays | Decays | ARMA(p,q) |

In [ ]:
# Generate AR(2) and MA(2) for comparison
ar2 = ArmaProcess([1, -0.7, 0.2], [1])
ar2_data = ar2.generate_sample(nsample=300)

ma2 = ArmaProcess([1], [1, 0.6, 0.3])
ma2_data = ma2.generate_sample(nsample=300)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# AR(2) - ACF and PACF
plot_acf(ar2_data, lags=20, ax=axes[0, 0])
axes[0, 0].set_title('AR(2): ACF decays gradually', fontweight='bold')

plot_pacf(ar2_data, lags=20, ax=axes[0, 1], method='ywm')
axes[0, 1].set_title('AR(2): PACF cuts off at lag 2', fontweight='bold')

# MA(2) - ACF and PACF
plot_acf(ma2_data, lags=20, ax=axes[1, 0])
axes[1, 0].set_title('MA(2): ACF cuts off at lag 2', fontweight='bold')

plot_pacf(ma2_data, lags=20, ax=axes[1, 1], method='ywm')
axes[1, 1].set_title('MA(2): PACF decays gradually', fontweight='bold')

plt.tight_layout()
plt.show()

## The "I": Integration (Differencing)

ARMA needs stationary data. If non-stationary:

**d = 1**: Take first difference  
**d = 2**: Take second difference (rare)

In [ ]:
# Generate non-stationary data (random walk)
drift = 0.5
innovations = np.random.normal(0, 1, 200)
random_walk = np.cumsum(innovations) + drift * np.arange(200)

# Test stationarity
adf_original = adfuller(random_walk)
print(f"Original ADF p-value: {adf_original[1]:.6f}")

if adf_original[1] > 0.05:
    print("→ Non-stationary! Differencing...")
    
    # First difference
    first_diff = np.diff(random_walk)
    adf_diff = adfuller(first_diff)
    print(f"After differencing ADF p-value: {adf_diff[1]:.6f}")
    
    if adf_diff[1] < 0.05:
        print("✓ Stationary! Use d=1")

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

axes[0].plot(random_walk, linewidth=2, color='#C73E1D')
axes[0].set_title('Original: Non-Stationary', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(first_diff, linewidth=2, color='#06A77D')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.3)
axes[1].set_title('After Differencing (d=1): Stationary', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Time')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Complete Workflow

![ARIMA Workflow](arima_workflow_flowchart.png)

**Process:**
1. Plot data
2. Test stationarity (ADF, KPSS)
3. If non-stationary, difference (d=1)
4. Plot ACF/PACF
5. Identify p and q
6. Fit ARIMA(p,d,q)
7. Diagnose residuals
8. Forecast

In [ ]:
# Real example: Exchange rate
n = 500
drift_fx = 0.01
shocks = np.random.normal(0, 0.5, n)
fx_rate = 100 + np.cumsum(drift_fx + shocks)

df_fx = pd.DataFrame({'rate': fx_rate}, 
                     index=pd.date_range('2020-01-01', periods=n, freq='D'))

# Step 1: Plot
plt.figure(figsize=(16, 5))
plt.plot(df_fx.index, df_fx['rate'], linewidth=1.5)
plt.title('Exchange Rate', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

# Step 2: Test stationarity
adf_test = adfuller(df_fx['rate'])
print(f"\nADF p-value: {adf_test[1]:.6f}")

if adf_test[1] > 0.05:
    print("Non-stationary → Take first difference")
    
    df_fx['rate_diff'] = df_fx['rate'].diff()
    
    # Step 3: Re-test
    adf_diff = adfuller(df_fx['rate_diff'].dropna())
    print(f"After diff ADF p-value: {adf_diff[1]:.6f}")
    
    if adf_diff[1] < 0.05:
        print("✓ Stationary! d=1")
        
        # Step 4: ACF/PACF
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        
        plot_acf(df_fx['rate_diff'].dropna(), lags=40, ax=axes[0])
        axes[0].set_title('ACF: Differenced Exchange Rate')
        
        plot_pacf(df_fx['rate_diff'].dropna(), lags=40, ax=axes[1], method='ywm')
        axes[1].set_title('PACF: Differenced Exchange Rate')
        
        plt.tight_layout()
        plt.show()
        
        print("\nNext: Identify p and q from patterns")
        print("Then fit ARIMA(p,1,q) in Part 6!")

## Key Takeaways

✓ **AR**: Past values → current value  
✓ **MA**: Past errors → current value  
✓ **I**: Differencing for stationarity  
✓ **ACF/PACF**: Tools for identifying p and q

**Patterns:**
- AR: PACF cuts off, ACF decays
- MA: ACF cuts off, PACF decays  
- ARMA: Both decay

## What's Next

**Part 6**: Implementation!
- Fitting ARIMA in Python
- Grid search for best (p,d,q)
- Diagnostics
- Forecasting with confidence intervals
- SARIMA for seasonality

Theory done. Now let's build models.